In [1]:
# Lab 3 Solution

import tensorflow as tf
import keras_hub
import numpy as np
import pandas as pd
import requests
from PIL import Image

# 1. Load ImageNet class names
classes_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_classes = requests.get(classes_url).text.strip().split("\n")
print("Num classes:", len(imagenet_classes))

# 2. Load KerasHub preprocessor + model 
preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
    "vit_base_patch16_224_imagenet"
)
model = keras_hub.models.ViTImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet"
)

# 3. 5 images path and true label
samples = [
    {"true_label": "golden retriever", "path": "/kaggle/input/datasets/layan1234layan/images-lab3/dog.jpg"},
    {"true_label": "tabby", "path": "/kaggle/input/datasets/layan1234layan/images-lab3/cat.jpg"},
    {"true_label": "banana", "path": "/kaggle/input/datasets/layan1234layan/images-lab3/bannana.jpg"},
    {"true_label": "sports car", "path": "/kaggle/input/datasets/layan1234layan/images-lab3/sport car.jpg"},
    {"true_label": "laptop", "path": "/kaggle/input/datasets/layan1234layan/images-lab3/laptop.png"},
]

def load_image_from_path(path: str):
    # PIL يدعم JPG/PNG
    img = Image.open(path).convert("RGB")
    img = np.array(img, dtype=np.uint8)
    return tf.convert_to_tensor(img, dtype=tf.uint8)

results = []

for i, s in enumerate(samples, start=1):
    img = load_image_from_path(s["path"])

    # Preprocess exactly as the preset expects
    x = preprocessor(img)
    x = tf.expand_dims(x, axis=0)  

    logits = model(x) 
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]

    top1 = int(np.argmax(probs))
    pred_name = imagenet_classes[top1]
    pred_conf = float(probs[top1])


    correct = s["true_label"].lower() in pred_name.lower()

    results.append({
        "Image File": i,
        "Predicted Label": pred_name,
        "True Label": s["true_label"],
        "Confidence": round(pred_conf, 4),
        "Correct?": "Yes" if correct else "No"
    })

df = pd.DataFrame(results)
df

2026-04-01 13:37:53.433658: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775050673.739704      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775050673.825900      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775050674.566804      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775050674.566857      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775050674.566861      55 computation_placer.cc:177] computation placer alr

Num classes: 1000


2026-04-01 13:38:35.090721: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


,Image File,Predicted Label,True Label,Confidence,Correct?
0,1,golden retriever,golden retriever,0.9541,Yes
1,2,tabby,tabby,0.4599,Yes
2,3,banana,banana,0.9962,Yes
3,4,sports car,sports car,0.9688,Yes
4,5,notebook,laptop,0.6327,No
